In [5]:
pip install opencv-python numpy cmake


  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl (39.0 MB)
Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl (12.6 MB)
   ---------------------------------------- 0.0/36.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.9 MB ? eta -:--:--
   ---------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.2.6 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.31.1 which is incompatible.


In [8]:
pip install dlib


  Using cached dlib-20.0.0.tar.gz (3.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build dlib
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [46 lines of output]
  running bdist_wheel
  running build
  running build_ext
  Traceback (most recent call last):
    File "<frozen runpy>", line 198, in _run_module_as_main
    File "<frozen runpy>", line 88, in _run_code
    File "F:\anaconda\Scripts\cmake.exe\__main__.py", line 4, in <module>
  ModuleNotFoundError: No module named 'cmake'
  
  
                     CMake is not installed on your system!
  
      Or it is possible some broken copy of cmake is installed on your system.
      It is unfortunately very common for python package managers to include
      broken copies of cmake.  So if the error above this refers to some file
      path to a cmake file inside a python or anaconda or miniconda path then you
      should delete that broken copy of cmake from your computer.
  
      Instead, please get an official copy of cmake from one of these known good
      sources of an official cmake:
          - cmake.org 

In [7]:
import cv2
import numpy as np
import dlib

# Load the images
img1 = cv2.imread("face1..jpeg")
img2 = cv2.imread("face2.jpeg")

# Resize images to same size (optional, for better alignment)
img1 = cv2.resize(img1, (500, 600))
img2 = cv2.resize(img2, (500, 600))

# Load face detector and facial landmarks predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# Get facial landmarks
def get_landmarks(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)
    if len(faces) == 0:
        raise Exception("No face detected.")
    return np.matrix([[p.x, p.y] for p in predictor(gray, faces[0]).parts()])

# Create face mask from landmarks
def get_face_mask(img, landmarks):
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    points = cv2.convexHull(landmarks)
    cv2.fillConvexPoly(mask, points, 255)
    return mask

# Warp source image onto destination
def warp_face(src_img, src_points, dest_points, dest_shape):
    warp_mat = cv2.getAffineTransform(np.float32(src_points[:3]), np.float32(dest_points[:3]))
    return cv2.warpAffine(src_img, warp_mat, (dest_shape[1], dest_shape[0]), None, flags=cv2.INTER_LINEAR)

# Swap faces
def swap_faces(img1, img2):
    points1 = get_landmarks(img1)
    points2 = get_landmarks(img2)

    hull1 = cv2.convexHull(points1)
    hull2 = cv2.convexHull(points2)

    mask1 = get_face_mask(img1, points1)
    mask2 = get_face_mask(img2, points2)

    face1 = cv2.bitwise_and(img1, img1, mask=mask1)
    face2 = cv2.bitwise_and(img2, img2, mask=mask2)

    r = cv2.boundingRect(hull2)
    center = (r[0] + r[2]//2, r[1] + r[3]//2)

    output = cv2.seamlessClone(face1, img2, mask1, center, cv2.NORMAL_CLONE)

    return output

# Perform face swap
swapped = swap_faces(img1, img2)

# Show and save result
cv2.imshow("Face Swap", swapped)
cv2.imwrite("face_swapped.jpg", swapped)
cv2.waitKey(0)
cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'dlib'